# Research runs: DistilBERT on 30k comments (Kaggle GPU)

Three DistilBERT runs on the same 30k training subset, each followed by per-label threshold tuning and the identity-term bias audit:

| Run | Loss | What it tests |
|---|---|---|
| A `distilbert_wbce_30k` | weighted BCE | Does a context model reduce the identity false positives that TF-IDF has? Does it react to Indian terms? |
| B `distilbert_wbce_idw3_30k` | weighted BCE, identity weight 3 | Vaidya et al. (2020) re-weighting on a transformer |
| C `distilbert_focal_30k` | focal loss | Focal vs weighted BCE for rare labels |

About 12–15 minutes per run on a T4. Section 5 (more seeds) is optional.

**Settings:** *Session options* → Accelerator **GPU T4 x2** or **P100**, Internet **On** (needs a phone-verified Kaggle account).
**Do not** use *Save & Run All* unless you want every cell to run. Download `research_runs.zip` from the *Output* panel at the end.

> Content note: the data contains offensive language.

In [ ]:
REPO_URL = "https://github.com/shriniwas2412/iiith-mml-explainable-toxicity.git"
BRANCH = "main"

import os
if not os.path.exists("repo"):
    !git clone -q -b {BRANCH} {REPO_URL} repo
%cd repo
!git pull -q
!pip install -q -e .
!git rev-parse --short HEAD

In [ ]:
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn the GPU accelerator on")

## 1. Data, splits and bias templates (about 3 minutes)

In [ ]:
!python scripts/01_download_data.py
!python scripts/03_make_splits.py --subset 30000
!python scripts/07_fetch_bias_resources.py
!python scripts/08_build_templates.py

## 2. Run A: weighted BCE

In [ ]:
RUN = "distilbert_wbce_30k"
!python scripts/05_train_transformer.py --train data/processed/train_subset_30000.csv --loss weighted_bce --out runs/{RUN}
!python scripts/06_tune_thresholds.py --run runs/{RUN}
!python scripts/09_bias_audit.py --model hf:runs/{RUN}/model --thresholds runs/{RUN}/thresholds.json --out reports/bias/{RUN}

## 3. Run B: weighted BCE + identity re-weighting (Vaidya et al., 2020)

In [ ]:
RUN = "distilbert_wbce_idw3_30k"
!python scripts/05_train_transformer.py --train data/processed/train_subset_30000.csv --loss weighted_bce --identity-weight 3 --out runs/{RUN}
!python scripts/06_tune_thresholds.py --run runs/{RUN}
!python scripts/09_bias_audit.py --model hf:runs/{RUN}/model --thresholds runs/{RUN}/thresholds.json --out reports/bias/{RUN}

## 4. Run C: focal loss

In [ ]:
RUN = "distilbert_focal_30k"
!python scripts/05_train_transformer.py --train data/processed/train_subset_30000.csv --loss focal --out runs/{RUN}
!python scripts/06_tune_thresholds.py --run runs/{RUN}
!python scripts/09_bias_audit.py --model hf:runs/{RUN}/model --thresholds runs/{RUN}/thresholds.json --out reports/bias/{RUN}

## 5. Optional: two more seeds of run A (no claim from a single run)

In [ ]:
for seed in (2, 3):
    RUN = f"distilbert_wbce_30k_seed{seed}"
    !python scripts/05_train_transformer.py --train data/processed/train_subset_30000.csv --loss weighted_bce --seed {seed} --out runs/{RUN}
    !python scripts/06_tune_thresholds.py --run runs/{RUN}
    !python scripts/09_bias_audit.py --model hf:runs/{RUN}/model --thresholds runs/{RUN}/thresholds.json --out reports/bias/{RUN}

## 6. Compare all runs and save the results

In [ ]:
!python scripts/12_compare_runs.py
!cat reports/compare/runs_table.md

In [ ]:
# Results without model weights (small). Download it from the Output panel on the right.
!zip -qr /kaggle/working/research_runs.zip runs reports -x "runs/*/model/*" -x "runs/*/model.joblib"
# Weights of run A, only if you need them for the demo or explanations (about 250 MB):
# !zip -qr /kaggle/working/distilbert_wbce_30k_model.zip runs/distilbert_wbce_30k/model
!ls -lh /kaggle/working/*.zip

**After downloading:** unzip `research_runs.zip` inside the repo folder on your laptop, so the files land in `runs/` and `reports/`, then run `python scripts/12_compare_runs.py` there to merge them with the laptop experiments. See `docs/RUN_GUIDE.md`.